In [ ]:
import os 
import re
import json
import logging
import pandas as pd 
import numpy as np
from typing import List, Dict, Any, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.vectorstores import Chroma
from backend.app.config import (
    OPENAI_KEY,
    MINI_LM_EMBED,
    GPT_4o,
    OPENAI_EMBED,
    VECTOR_DB_PATH,
    PROJECT_ROOT
)
DEFAULT_EMBED_MODEL = OPENAI_EMBED

In [65]:
question_answer_pairs = [
  {
    "question": "Which further imaging is indicated for persistent hip pain over 6 weeks despite normal X-ray findings?",
    "answer_reference": "Recommendation: MRI for persistent symptoms with normal X-ray findings."
  },
  {
    "question": "When should an MRI of both hip joints be performed?",
    "answer_reference": "Recommendation: MRI of both hips for unilateral femoral head necrosis in ARCO stages I-IV."
  },
  {
    "question": "Which classification is recommended for staging atraumatic femoral head necrosis?",
    "answer_reference": "Recommendation: Use of the modified ARCO classification."
  },
  {
    "question": "What should be done if a subchondral fracture in ARCO Stage II is suspected, but the diagnosis is unclear?",
    "answer_reference": "Recommendation: Perform a CT scan to clarify the subchondral fracture."
  },
  {
    "question": "Should scintigraphy be used to diagnose atraumatic femoral head necrosis?",
    "answer_reference": "Recommendation: Scintigraphy is not recommended for the diagnosis of atraumatic femoral head necrosis."
  },
  {
    "question": "How does one differentiate between transient bone marrow edema and osteonecrosis on MRI?",
    "answer_reference": "Recommendation: MRI patterns and clinical course are crucial for differentiation."
  },
  {
    "question": "Which imaging method is considered the gold standard for diagnosing atraumatic femoral head necrosis?",
    "answer_reference": "Recommendation: MRI as the gold standard."
  },
  {
    "question": "Which imaging technique is best suited for detecting a subchondral fracture?",
    "answer_reference": "Recommendation: CT for visualizing subchondral fractures."
  },
  {
    "question": "What risk factors indicate bilateral involvement in femoral head necrosis?",
    "answer_reference": "Recommendation: Unilateral femoral head necrosis increases the risk of bilateral disease; consider risk factors."
  },
  {
    "question": "What radiological findings characterize ARCO Stage III?",
    "answer_reference": "Recommendation: Signs of a subchondral fracture with incipient articular surface incongruity on the radiograph."
  }
]

### Document Extraction & Splitting

In [5]:
def extract_text_from_pdf(file_path: str) -> str:
        """
        Extract text from a PDF file using PyPDFLoader.
        """
        try:
            loader = PyPDFLoader(file_path)
            documents = loader.load()
            return [page.page_content for page in documents]
        except Exception as e: 
            raise Exception(f"Error extracting text from PDF: {e}")
        
class CleanText:
    """Simplified text cleaner for medical documents."""
    
    def __init__(self, text: str):
        self.text = text
        
        # Common patterns to remove
        self.header_patterns = [
            r'^S3-Leitlinie.*?Langfassung\s+Version vom \d{2}\.\d{2}\.\d{4}.*?\n',
            r'^Seite \d+ von \d+'
        ]
        self.footer_patterns = [
            r'\n\d+\s*$',  # Page numbers at end
            r'©.*?$'       # Copyright notices
        ]
        
    def remove_unwanted_patterns(self) -> 'CleanText':
        """Remove headers, footers, and other unwanted patterns."""
        all_patterns = self.header_patterns + self.footer_patterns
        for pattern in all_patterns:
            self.text = re.sub(pattern, '', self.text, flags=re.MULTILINE)
        return self
    
    def clean_special_chars(self) -> 'CleanText':
        """Remove unusual characters while keeping basic punctuation."""
        # Keep letters, numbers, basic punctuation, and whitespace
        self.text = re.sub(r'[^\w\s.,;:\-()/°]', '', self.text)
        return self
    
    def fix_spacing(self) -> 'CleanText':
        """Clean up spacing and line breaks."""
        # Replace multiple spaces with single space
        self.text = re.sub(r' +', ' ', self.text)
        # Replace single newlines with space (keep paragraphs)
        self.text = re.sub(r'(?<!\n)\n(?!\n)', ' ', self.text)
        # Remove space around hyphens
        self.text = re.sub(r'\s*-\s*', '-', self.text)
        return self
    
    def clean(self) -> str:
        """Run the complete cleaning process."""
        return (
            self.remove_unwanted_patterns()
            .clean_special_chars()
            .fix_spacing()
            .text
        )



In [9]:
pdf_path = os.path.join(os.getcwd(), "backend", "app", "documents", "Guideline_atraumatische_Femurkopfnekrose_2019-09_1-abgelaufen.pdf")

pdf_extraction = extract_text_from_pdf(pdf_path)
print(pdf_extraction)

if isinstance(pdf_extraction, list):
    pdf_extraction = "\n".join(pdf_extraction)

cleaner = CleanText(pdf_extraction)
cleaned_text = cleaner.clean()
print(cleaned_text)




['S3-Leitlinie Atraumatische Femurkopfnekrose des Erwachsenen - Langfassung \nVersion vom 18.09.2019 1 \nLangfassung \nS3-Leitlinie „Atraumatische Femurkopfnekrose des Erwachsenen“  \nVersion 2.4 – August 2019 \nAWMF-Register-Nr. 033/050 – Atraumatische Femurkopfnekrose des Erwachsenen \nFederführender Autor  \nProf. Dr. A. Roth \nFederführende Fachgesellschaft  \nDeutsche Gesellschaft für Orthopädie und Orthopädische Chirurgie (DGOOC) \nBeteiligte Fachgesellschaften \nDeutsche Gesellschaft für Unfallchirurgie (DGU) \nDeutsche Gesellschaft für Muskuloskelettale Radiologie (DGMSR) \nDeutschen Gesellschaft für Physikalische Medizin und Rehabilitation (DGPMR) \nDachverband Osteologie (DVO) \nRheumaliga Bundesverband e.V. (DRL) \npubliziert bei:', 'S3-Leitlinie Atraumatische Femurkopfnekrose des Erwachsenen - Langfassung \n \nVersion vom 18.09.2019  2 \n \nINHALTSVERZEICHNIS  \n \n1. Geltungsbereich und Zweck ....................................................................... 4 \nBegrü

### Text Splitting

In [20]:
def split_text(
        pdf_docs: List[str], 
        chunk_size: int = 1000, 
        chunk_overlap: int = 200) -> List[Document]:
    """
    Splits text into chunks for efficient embedding.
    
    - `chunk_size`: Max characters per chunk.
    - `chunk_overlap`: Overlap between chunks for better context retention.
    
    Returns a list of text chunks.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len
    )

    chunks = []
    for page_text in pdf_docs:
        split_chunks = text_splitter.split_text(page_text)
        for chunk in split_chunks:
            chunks.append(Document(page_content=chunk))
    
    return chunks



In [30]:
text_chunks = split_text([cleaned_text], chunk_size=500, chunk_overlap=100)
print(f"Number of chunks: {len(text_chunks)}")

Number of chunks: 945


In [33]:
print(text_chunks[0].page_content)

Langfassung  S3-Leitlinie Atraumatische Femurkopfnekrose des Erwachsenen  Version 2.4 August 2019  AWMF-Register-Nr. 033/050 Atraumatische Femurkopfnekrose des Erwachsenen  Federführender Autor  Prof. Dr. A


### Word Embeddings & Vector Storage

In [39]:
def store_embeddings(
        docs: List[Document],
        embed_model: Optional[str] = None,
        collection_name: str = "chromadb"
    ) -> None:

    """
    Generates embeddings for text chunks and stores them in ChromaDB.

    Args:
        docs (List[Document]): List of documents to embed.
        embed_model (Optional[str]): Name of the embedding model.  Defaults to OPENAI_EMBED.
        collection_name (str): Name of the ChromaDB collection.  Defaults to "chromadb".
    """

    if embed_model is None:
        embed_model = OPENAI_EMBED

    logging.info(f"Using embedding model: {embed_model}") 

    if embed_model == OPENAI_EMBED:
        embedding_model = OpenAIEmbeddings(model=OPENAI_EMBED, api_key=OPENAI_KEY)
    elif embed_model == MINI_LM_EMBED:
        embedding_model = HuggingFaceEmbeddings(model_name=MINI_LM_EMBED)
    else:
        raise ValueError(f"Unsupported embedding model: {embed_model}") 
    
    text_chunks = [doc.page_content for doc in docs]
    metadata = [doc.metadata for doc in docs]
    logging.info(f"Number of chunks to embed: {len(text_chunks)}") 

    dir_path = os.path.join(VECTOR_DB_PATH, collection_name)
    os.makedirs(dir_path, exist_ok=True)
    logging.info(f"ChromaDB directory: {dir_path}")

    db = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_model, 
        persist_directory=dir_path
    )
    db.add_texts(texts=text_chunks, metadatas=metadata, embeddings=embedding_model) 

    logging.info(f"✅ Stored {len(text_chunks)} text chunks in ChromaDB (Collection: {collection_name})")

In [40]:
store_embeddings(text_chunks, embed_model=OPENAI_EMBED, collection_name="medical_dataset_test")

/var/folders/7t/g4h6fbw915v56stlfs263jzc0000gn/T/ipykernel_44661/4253828258.py:36: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


In [42]:
def retrieve_text(
        vectordb_path: str,
        query: str,
        embed_model: Optional[str] = None,
        collection_name: str = "chromadb",
        results_to_return: int = 3):
    """
    Retrieve text from the vector store based on the query.

    Args:
        vector_db (Chroma): ChromaDB instance.
        query (str): Query string to search for in the vector store.

    Returns:
        List[Document]: Retrieved documents from the vector store.
    """
    if not os.path.exists(vectordb_path):
        raise FileNotFoundError(f"Vector store not found at: {vectordb_path}")

    if not query:
        raise ValueError("Query cannot be empty.")

    if embed_model is None:
        embed_model = DEFAULT_EMBED_MODEL

    embedding_model = OpenAIEmbeddings(model=embed_model, api_key=OPENAI_KEY)

    print(f"Retrieving from Vector DB Path: {vectordb_path}")
    vector_db = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_model,
        persist_directory=vectordb_path
    )

    matched_texts = vector_db.similarity_search_with_score(query, k=results_to_return)

    print(f"Number of Retrieved Texts: {len(matched_texts)}")
    print(f"Query used for retrieval: {query}")

    return matched_texts

In [46]:
vectordb_path = os.path.join(VECTOR_DB_PATH, "medical_dataset_test")
question = "weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?"
answer_reference = ["MRT bei persistierenden Beschwerden mit unauffälligem Röntgenbefund"] 

matched_texts = retrieve_text(
        vectordb_path=vectordb_path,
        query=question,
        embed_model=OPENAI_EMBED,
        collection_name="medical_dataset_test",
        results_to_return=3
    )

for text, score in matched_texts:
    print(f"Score: {score}", end="\n")
    print(f"Text: {text.page_content}", end="\n")
    print(f"Metadata: {text.metadata}", end="\n")



Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset_test
Number of Retrieved Texts: 3
Query used for retrieval: weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?
Score: 0.578153669834137
Text: . Hinken und Bewegungsschmerz bzw.  Bewegungseinschränkungen, kein Hinweis auf Di fferentialdiagnosen) soll  zunächst eine Röntgenuntersuchung (Beckenübersicht und betroffene Hüfte in  Lauensteinprojektion) durchgeführt werden.  Expertenkonsens (stark) Entscheidung basiert aus-schließlich auf Experten-konsens, 100  Zustimmung  Bei unauffälligem Röntgenbild und anhaltenden Beschwerden soll eine MRT des  Hüftgelenkes veranlasst werden
Metadata: {}
Score: 0.6730622053146362
Text: . Es ist der SPECT überlegen. Die Autoren ermittelten,  daß der Schmerz der Hauptgrund für die Untersuchung war. Sie schlussfolgerten, daß bei  klinischem Verdacht auf Femurkopfnekrose (anhaltender Leistenschmerz) und normalen  Rön

### Chatbot

In [ ]:
class Chatbot:
    def __init__(self, 
                model_type: str = 'openai',
                temperature: float = 0.2,
                max_tokens: int = 100,
                vectordb_path: str = os.path.join(VECTOR_DB_PATH, "medical_dataset_test"),
                collection_name: str = "medical_dataset_test",
                top_p: float = 0.95):
        """
        Initializes the chatbot with medical QA capabilities.

        Args:
            model_type (str): The type of model to use ('openai' or 'huggingface').
        """
        self.model_type = model_type
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.top_p = top_p
        self.vectordb_path = vectordb_path
        self.collection_name = collection_name

        self.llm = self.initialize_model()
        self.retriever = lambda query: retrieve_text(
            vectordb_path=self.vectordb_path,
            query=query,
            collection_name=self.collection_name
        )

    def initialize_model(self):
        """Initialize the LLM with medical-focused parameters"""
        return ChatOpenAI(
            model=GPT_4o,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
            top_p=self.top_p,
            api_key=OPENAI_KEY
        )

    def ask(self, question: str) -> str:
        """
        Ask a question to the chatbot and get an answer.

        Args:
            question (str): The question to ask.

        Returns:
            str: The answer from the chatbot.
        """
        if not question:
            raise ValueError("Question cannot be empty.")

        # Retrieve relevant documents
        matched_texts = self.retriever(question)

        if not matched_texts:
            return "I don't know (no relevant documents found)."
        
        context = "\n".join([doc.page_content for doc, _ in matched_texts])
        
        prompt = ChatPromptTemplate.from_template("""
            Answer this medical question based ONLY on the context below.
            Be concise and factual. If unsure, say 'I don't know'.

            Question: {input}
            Context: {context}

            Answer in the same language as the question:
        """)
        
        formatted_prompt = prompt.format(input=question, context=context)
        
        response = self.llm.invoke(formatted_prompt)
        return response.content

In [55]:
question = "weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?"

bot = Chatbot()
answer = bot.ask(question)
print(answer)

Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset_test
Number of Retrieved Texts: 3
Query used for retrieval: weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?
Ja, weiterführende Bildgebung, wie ein MRT des Hüftgelenkes, ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund.


### Accuracy

/Users/syedalimuradtahir/.pyenv/versions/3.10.8/lib/python3.10/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


AttributeError: 'Client' object has no attribute 'trace'